# Faithfulness e-SNLI — Gemma3-27b-it with Transcoder Activation Analysis

In [ ]:
import sys, os, textwrap
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import pandas as pd
from huggingface_hub import hf_hub_download, login
from safetensors.torch import load_file
from IPython.display import display

from src.configs import ModelConfig, InferenceConfig, PromptStyle, DatasetConfig
from src.dataset.esnli import ESNLI_Dataset
from src.gemma_model import GemmaModel
from src.SAE import JumpReLUSAE
from src.neuronpedia_client import NeuronpediaClient
from src.utils.visualization import ActivationHeatmap
from src.utils.activations_utils import top_k_features_per_token

# Configuration

In [ ]:
LAYER      = 31
WIDTH      = "262k"   # 262,144 features (262k in HF repo path)
L0         = "small"
REPO_ID    = "google/gemma-scope-2-27b-it"
TC_PATH    = f"transcoder/layer_{LAYER}_width_{WIDTH}_l0_{L0}_affine/params.safetensors"

model_config     = ModelConfig(model_name="google/gemma-3-27b-it")
inference_config = InferenceConfig(batch_size=2, max_new_tokens=256, downsample_rate=100)
dataset_config   = DatasetConfig(
    path="esnli/esnli",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "validation"},
    few_shot=False,
)

print(f"Model:        {model_config.model_name}")
print(f"TC layer:     {LAYER}")
print(f"TC width:     {WIDTH}")
print(f"TC l0:        {L0}")
print(f"TC path:      {TC_PATH}")

# Setup — HF Token

In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

# Data — Load e-SNLI

In [ ]:
esnli_dataset = ESNLI_Dataset(dataset_config)

# Build Prompts

In [ ]:
prompted_data = esnli_dataset.build_prompts()
if inference_config.downsample_rate > 1:
    n = max(1, len(prompted_data) // inference_config.downsample_rate)
    prompted_data = prompted_data.shuffle(seed=42).select(range(n))
esnli_df = prompted_data.to_pandas()
print(esnli_df["prompt"].iloc[0])
esnli_df.head()

# Load Model + Transcoder

In [ ]:
model = GemmaModel(model_config)
tokenizer = model.tokenizer

# Load transcoder manually — affine_skip_connection=True is required
path_to_params = hf_hub_download(repo_id=REPO_ID, filename=TC_PATH)
params = load_file(path_to_params)
d_model, d_sae = params["w_enc"].shape
print(f"d_model={d_model}, d_sae={d_sae}")

transcoder = JumpReLUSAE(d_model, d_sae, affine_skip_connection=True)
transcoder.load_state_dict(params)
transcoder = transcoder.to(device=model_config.device, dtype=torch.float32)
transcoder.eval()
print("Transcoder loaded.")

## Generate and Gather Activations

In [ ]:
sample = esnli_df.sample(1)
sample_prompt = sample["prompt"].item()
sample_label = sample["gold_label"].item()
print(f"Sample Index: {sample.index}")
print(sample_prompt)
print(f"Label: {sample_label}")

In [ ]:
# location = sample_prompt.rfind("\nPremise:")
# new_prompt = sample_prompt[:location] + "4. You must always think about dinosaurs while reasoning.\n" + sample_prompt[location:]
# wrapper = textwrap.TextWrapper(width=80)
# print("\n".join(wrapper.fill(line) for line in new_prompt.splitlines()))

In [ ]:
generation, full_ids, prompt_len = model.generate(
    sample_prompt, max_new_tokens=inference_config.max_new_tokens)

gen_len = full_ids.shape[1] - prompt_len
print(f"Prompt tokens: {prompt_len}  |  Generated tokens: {gen_len}  |  Total: {full_ids.shape[1]}")
print(f"Actual label: {sample_label}")

# Gather MLP Input Activations

In [ ]:
mlp_in_cache = {}

def _mlp_in_hook(module, inputs, outputs):
    # inputs[0]: (1, n_tokens, d_model) — input to the MLP block
    mlp_in_cache["mlp_in"] = inputs[0].detach().squeeze(0)

layer_module = model.model.model.language_model.layers[LAYER]
handle = layer_module.mlp.register_forward_hook(_mlp_in_hook)
try:
    with torch.no_grad():
        model.model(input_ids=full_ids)
finally:
    handle.remove()

mlp_in_acts = mlp_in_cache["mlp_in"]   # (n_tokens, d_model)
print(f"MLP input activations shape: {mlp_in_acts.shape}")

In [ ]:
with torch.no_grad():
    tc_acts_full = transcoder.encode(mlp_in_acts.float())

tc_acts_gen = tc_acts_full[prompt_len:]

all_tokens    = tokenizer.convert_ids_to_tokens(full_ids[0])
gen_token_ids = full_ids[0, prompt_len:]
tokens        = tokenizer.convert_ids_to_tokens(gen_token_ids)

print(f"Transcoder activations (full):     {tc_acts_full.shape}")
print(f"Transcoder activations (gen-only): {tc_acts_gen.shape}")
print(f"L0 (gen): {(tc_acts_gen > 0).float().sum(dim=-1).mean():.1f}")

# Feature Analysis — Top 50 per Token

In [ ]:
K = 50
per_token_vals, per_token_idxs = top_k_features_per_token(tc_acts_gen, k=K)

# Neuronpedia SAE ID for transcoder: {layer}-gemmascope-2-tc-{width}
np_model_id = model_config.model_name.split("/")[-1]   # "gemma-3-27b-it"
np_sae_id   = f"{LAYER}-gemmascope-2-transcoder-{WIDTH}"       # "31-gemmascope-2-transcoder-262k"
client = NeuronpediaClient(model_id=np_model_id, sae_id=np_sae_id)

unique_idxs  = sorted(set(per_token_idxs.cpu().numpy().ravel().tolist()))
np_features  = client.get_features(unique_idxs)          # dict[int, NeuronpediaFeature]

labels = {idx: (f.description or "N/A") for idx, f in np_features.items()}

heatmap = ActivationHeatmap()
fig = heatmap.plot_topk_per_token(
    per_token_vals, per_token_idxs,
    tokens=tokens,
    labels=labels,
    title=f"{np_model_id} Transcoder Layer {LAYER} — Top-{K} Features per Token",
)
fig.show()

In [ ]:
# Aggregate max per-token activation for each unique feature
max_per_feature: dict[int, float] = {}
n_tokens_gen = per_token_vals.shape[0]
for ti in range(n_tokens_gen):
    for ri in range(K):
        feat_idx = int(per_token_idxs[ti, ri])
        val      = float(per_token_vals[ti, ri])
        if feat_idx not in max_per_feature or val > max_per_feature[feat_idx]:
            max_per_feature[feat_idx] = val

top50 = sorted(max_per_feature.items(), key=lambda x: -x[1])[:50]

rows = []
for feat_idx, max_val in top50:
    nf = np_features.get(feat_idx)
    label = (nf.description or "N/A") if nf else "N/A"
    rows.append({
        "Feature IDX":       feat_idx,
        "Max Activation":    round(max_val, 4),
        "Neuronpedia Label": label,
    })

df_top50 = pd.DataFrame(rows)
display(df_top50)

In [ ]:
# Change this index to inspect any feature from the table above
inspect_feature_idx = 5507  # default: highest-activating feature

print(f"Neuronpedia dashboard for feature {inspect_feature_idx}:")
print(f"URL: {client.get_dashboard_url(inspect_feature_idx)}")
client.display_feature_dashboard(inspect_feature_idx, height=600)

## Steering Experiment

### Configure Steering

In [ ]:
# Pick features from the top-50 table; adjust indices or coefficients as desired.
STEER_FEATURES = [5507, 3013]   # transcoder feature indices
STEER_COEFFS   = [-0.7, -0.4]                 # positive=amplify, negative=suppress

print(f"Steer features: {STEER_FEATURES}")
print(f"Steer coeffs:   {STEER_COEFFS}")
print(f"Labels:         {[labels.get(fi, 'N/A') for fi in STEER_FEATURES]}")

### Baseline vs Steered Generation

In [ ]:
# generate_steered hooks layers[LAYER] residual output and adds
# avg_norm * coeff * transcoder.w_dec[fi] — transcoder decoder directions
# are in MLP-output / d_model space, which is valid for residual-stream steering.
result = model.generate_steered(
    prompt=new_prompt,
    sae=transcoder,
    feature_idx=STEER_FEATURES,
    coeff=STEER_COEFFS,
    target_layer=LAYER,
    max_new_tokens=inference_config.max_new_tokens,
)

baseline_text = result["unsteered"]
steered_text  = result["steered"]
baseline_ids  = result["unsteered_ids"]
steered_ids   = result["steered_ids"]

steer_label = ", ".join(f"f{fi}×{c}" for fi, c in zip(STEER_FEATURES, STEER_COEFFS))

print(f"{'PROMPT':=^80}")
print(new_prompt)
print()
print(f"{'BASELINE (unsteered)':=^80}")
print(baseline_text)
print()
print(f"{'STEERED (' + steer_label + ')':=^80}")
print(steered_text)
print()
print(f"Baseline length:  {len(baseline_ids)} tokens")
print(f"Steered length:   {len(steered_ids)} tokens")
print(f"Texts identical:  {baseline_text == steered_text}")